# QA — Reranking Results Inspection

Lightweight QA tool for inspecting reranking results.
Tables with product images, side-by-side baseline vs reranker, and a single NDCG comparison table.

In [6]:
import pandas as pd
import numpy as np
from IPython.display import display, HTML

import pandas as pd
import numpy as np
from IPython.display import display, HTML
import sys
from pathlib import Path

root = Path.cwd().resolve()
if not (root / "src").exists() and (root.parent / "src").exists():
    root = root.parent
sys.path.insert(0, str(root))

from src.data.loader import load_search_data, load_product_metadata
from src.data.cleaner import clean_search_data, clean_product_metadata
from src.models.reranker import rerank
from src.models.strategies import RerankStrategy
from src.evaluation.metrics import evaluate_against_baseline

CDN_BASE = "https://cdn.aboutstatic.com/file"
from src.data.cleaner import clean_search_data, clean_product_metadata
from src.models.reranker import rerank
from src.models.strategies import RerankStrategy
from src.evaluation.metrics import evaluate_against_baseline

CDN_BASE = "https://cdn.aboutstatic.com/file"

## Load data

In [7]:
search_df = clean_search_data(load_search_data())
product_df = clean_product_metadata(load_product_metadata())
prod_lookup = product_df.set_index("product_id")

print(f"{len(search_df):,} rows | {search_df['search_term'].nunique()} terms | {len(product_df):,} products")

2026-06-07 21:37:59 | src.data.loader      | INFO    | Loading search data: D:\AboutYou\data\search_term_products.parquet
2026-06-07 21:38:00 | src.data.loader      | INFO    | Loaded search_data: 19129 rows, 9 columns, 1.63MB
2026-06-07 21:38:00 | src.data.cleaner     | INFO    | Starting cleaning pipeline: 19129 rows
2026-06-07 21:38:00 | src.data.cleaner     | INFO    | Filled 102 null impression_pos values with median=26.0
2026-06-07 21:38:00 | src.data.cleaner     | INFO    | Flagged 51 rows with CTR > 100% (legitimate: bookmarks/notifications)
2026-06-07 21:38:00 | src.data.cleaner     | INFO    | Flagged 7339 rows with impressions < 3 (low confidence)
2026-06-07 21:38:00 | src.data.cleaner     | INFO    | Cleaning pipeline complete: 19129 rows
2026-06-07 21:38:00 | src.data.loader      | INFO    | Loading product metadata: D:\AboutYou\data\products.csv
2026-06-07 21:38:00 | src.data.loader      | INFO    | Loaded product_metadata: 16697 rows, 4 columns, 2.04MB
2026-06-07 21:38:0

## Baseline vs Reranker (NDCG@10)

In [8]:
evaluate_against_baseline(search_df, k=10)

2026-06-07 21:38:04 | src.evaluation.metrics | INFO    | Evaluated baseline (sort by impression_pos_avg, ascending): 305 terms, mean NDCG@10=0.4193
2026-06-07 21:38:06 | src.evaluation.metrics | INFO    | Evaluated strategy=smoothed_ctr: 305 terms, mean NDCG@10=0.8781
2026-06-07 21:38:06 | src.evaluation.metrics | INFO    | 
=== Smoothed CTR vs Baseline (NDCG@10) ===
               strategy  mean_ndcg  std_ndcg  min_ndcg  max_ndcg  n_terms
           smoothed_ctr   0.878149  0.169871  0.098039       1.0      305
baseline_impression_pos   0.419291  0.310423  0.000000       1.0      305


,strategy,mean_ndcg,std_ndcg,min_ndcg,max_ndcg,n_terms
0,smoothed_ctr,0.878149,0.169871,0.098039,1.0,305
1,baseline_impression_pos,0.419291,0.310423,0.000000,1.0,305


## Inspect reranking for a query

Shows top products as the reranker orders them, with thumbnails, scores, and the original baseline rank.

In [10]:
def image_tag(image_hash, size=64):
    if pd.isna(image_hash) or not image_hash:
        return ""
    return f"<img src='{CDN_BASE}/{image_hash}?quality=75&height={size}&width={size}' width='{size}' height='{size}'>"


def inspect(term, k=10):
    """Show reranker results with product images and baseline rank delta."""
    term_df = search_df[search_df["search_term"] == term].copy()
    if term_df.empty:
        print(f"Term '{term}' not in dataset.")
        return

    # Baseline order
    term_df = term_df.sort_values("impression_pos_avg", ascending=True).reset_index(drop=True)
    term_df["baseline_rank"] = range(1, len(term_df) + 1)

    # Reranker order
    ranked = rerank(term, search_df, strategy=RerankStrategy.SMOOTHED_CTR, top_k=k)
    reranked_map = {r["product_id"]: r for r in ranked}

    rows = []
    for r in ranked:
        pid = r["product_id"]
        prod = prod_lookup.loc[pid] if pid in prod_lookup.index else pd.Series()
        baseline_row = term_df[term_df["product_id"] == pid]
        base_rank = int(baseline_row["baseline_rank"].iloc[0]) if not baseline_row.empty else "-"
        delta = base_rank - r["rank"] if isinstance(base_rank, int) else "-"
        delta_str = f"+{delta}" if isinstance(delta, int) and delta > 0 else str(delta)

        rows.append({
            "": image_tag(prod.get("image_hash")),
            "product": prod.get("product_name", ""),
            "rerank #": r["rank"],
            "baseline #": base_rank,
            "delta": delta_str,
            "score": r["score"],
            "clicks": r["clicks"],
            "impr.": r["impressions"],
        })

    header = f"<h3><code>{term}</code> — top {k} (smoothed CTR)</h3>"
    header += "<p>delta = baseline_rank - reranker_rank (positive = moved up)</p>"
    display(HTML(header + pd.DataFrame(rows).to_html(escape=False, index=False)))


# --- Try some queries ---
inspect("kleid", k=10)

Term 'kleid' not in dataset.


In [11]:
inspect("barrel jeans", k=10)

,product,rerank #,baseline #,delta,score,clicks,impr.
,jeans 'onlhorseshoe',1,98,+97,22.5711,229,979
,jeans 'bet',2,42,+40,20.2089,444,2162
,jeans 'vmbillie',3,4,+1,17.7833,1804,10110
,jeanshose baarell,4,16,+12,17.6654,226,1245
,'cinch barrel jeans',5,19,+14,17.0800,488,2823
,jeans 'danielle',6,87,+81,16.5497,282,1670
,jeans 'onltamy',7,38,+31,16.0693,546,3364
,jeans 'cinch barrel',8,3,-5,14.3864,1448,10032
,jeans 'jdysusie',9,2,-7,14.2542,1615,11297
,jeans 'baarell',10,79,+69,13.7368,142,1001


In [18]:
inspect("Sneaker 'Uno'", k=10)

Term 'Sneaker 'Uno'' not in dataset.


## Pick your own query

In [20]:
# Change the term below to inspect any query
inspect("Shapewear", k=10)

Term 'Shapewear' not in dataset.
